In [11]:
print("⏳ Preprocessing data pipeline for different models... \n")
print("🔗 Forming base dataset for all models...\n")
print("🕓 Normalizing the data...")

base_df = df.rename(columns={1: "time_step"})
base_df.columns = list(base_df.columns[:2]) + [f"f{i}" for i in range(2, 167)] + [base_df.columns[-1]]

print(f"✅ Data normalizing in base dataset is done.\nTotal number of transactions: {len(base_df)}\n")
print(base_df.head(3))

⏳ Preprocessing data pipeline for different models... 

🔗 Forming base dataset for all models...

🕓 Normalizing the data...
✅ Data normalizing in base dataset is done.
Total number of transactions: 203769

        txId  time_step        f2        f3        f4       f5        f6  \
0  230425980          1 -0.171469 -0.184668 -1.201369 -0.12197 -0.043875   
1    5530458          1 -0.171484 -0.184668 -1.201369 -0.12197 -0.043875   
2  232022460          1 -0.172107 -0.184668 -1.201369 -0.12197 -0.043875   

         f7        f8        f9  ...      f158      f159      f160      f161  \
0 -0.113002 -0.061584 -0.162097  ... -0.600999  1.461330  1.461369  0.018279   
1 -0.113002 -0.061584 -0.162112  ...  0.673103 -0.979074 -0.978556  0.018279   
2 -0.113002 -0.061584 -0.162749  ...  0.439728 -0.979074 -0.978556 -0.098889   

       f162      f163      f164      f165      f166    class  
0 -0.087490 -0.131155 -0.097524 -0.120613 -0.119792  unknown  
1 -0.087490 -0.131155 -0.097524 -0.120613 

In [12]:
print("🌲 Creating the Random Forest dataset...\n")
print("🔍Filtering out the unknown class transactions...")
random_forest_df = base_df[base_df["class"] != "unknown"].copy()
print("✅ Filtering is done...\n\nProceeding to creation of 'is_illicit' binary series and final cleaning...")

random_forest_df["is_illicit"] = (random_forest_df["class"] == "1").astype(int)
print("✅ Binary transaction class is created(Labels).\n")

print("🗑️Removing unnecessary series for an RF...\n")
random_forest_df.drop(columns=["class"], errors="ignore", inplace=True)
print(random_forest_df.head(3))

🌲 Creating the Random Forest dataset...

🔍Filtering out the unknown class transactions...
✅ Filtering is done...

Proceeding to creation of 'is_illicit' binary series and final cleaning...
✅ Binary transaction class is created(Labels).

🗑️Removing unnecessary series for an RF...

         txId  time_step        f2        f3        f4         f5        f6  \
3   232438397          1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
9   232029206          1 -0.005027  0.578941 -0.091383   4.380281 -0.063725   
10  232344069          1 -0.147852 -0.184668 -1.201369  -0.121970 -0.043875   

          f7         f8        f9  ...      f158      f159      f160  \
3   9.782742  12.414558 -0.163645  ... -0.613614  0.241128  0.241406   
9   4.667146   0.851305 -0.163645  ... -0.613614  0.241128  0.241406   
10 -0.113002  -0.061584 -0.137933  ... -0.613614  0.241128  0.241406   

        f161      f162      f163      f164      f165      f166  is_illicit  
3   1.072793  0.085530 -0.131155  0.

In [13]:
print("🛠️ Creating the unsupervised dataset...\n")
unsupervised_df = base_df.drop(columns=["time_step", "class"])
print(unsupervised_df.head(3))

🛠️ Creating the unsupervised dataset...

        txId        f2        f3        f4       f5        f6        f7  \
0  230425980 -0.171469 -0.184668 -1.201369 -0.12197 -0.043875 -0.113002   
1    5530458 -0.171484 -0.184668 -1.201369 -0.12197 -0.043875 -0.113002   
2  232022460 -0.172107 -0.184668 -1.201369 -0.12197 -0.043875 -0.113002   

         f8        f9       f10  ...      f157      f158      f159      f160  \
0 -0.061584 -0.162097 -0.167933  ... -0.562153 -0.600999  1.461330  1.461369   
1 -0.061584 -0.162112 -0.167948  ...  0.947382  0.673103 -0.979074 -0.978556   
2 -0.061584 -0.162749 -0.168576  ...  0.670883  0.439728 -0.979074 -0.978556   

       f161      f162      f163      f164      f165      f166  
0  0.018279 -0.087490 -0.131155 -0.097524 -0.120613 -0.119792  
1  0.018279 -0.087490 -0.131155 -0.097524 -0.120613 -0.119792  
2 -0.098889 -0.106715 -0.131155 -0.183671 -0.120613 -0.119792  

[3 rows x 166 columns]


In [14]:
try:
    import joblib
    from datetime import datetime
    from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
    )
    
    from sklearn.ensemble import RandomForestClassifier, IsolationForest
    from sklearn.neighbors import LocalOutlierFactor
    
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )
    
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    
    from scipy.stats import randint, uniform

    print("✅Libraries imported successfully.")
    
except:
    print("❌The error during import has occured")


✅Libraries imported successfully.


In [15]:
print("✂️ Train/test splitting for Random Forest dataset..")
rf_train_df, rf_test_df = train_test_split(
    random_forest_df,
    test_size=0.2,
    random_state=42,
    stratify=random_forest_df["is_illicit"]
)
if len(random_forest_df) == (len(rf_train_df) + len(rf_test_df)):
    print("✅ Splitting is done successfully\n")
else:
    print("🚨 Error has occured during splitting")

print(f"Train size: {rf_train_df.shape}")
print(f"Test size: {rf_test_df.shape}\n")

print("📥 Saving test transaction IDs")
test_txids = rf_test_df["txId"].values
print(f"✅Saved: {test_txids[:3]}...\n")

print("🛠️ Preparing Random Forest data...\n")
X_train_rf = rf_train_df.drop(columns=["txId", "time_step","is_illicit"], errors="ignore")
y_train_rf = rf_train_df["is_illicit"]

X_test_rf = rf_test_df.drop(columns=["txId","time_step","is_illicit"], errors="ignore")
y_test_rf = rf_test_df["is_illicit"]

print(f"📐 Train features' shape: {X_train_rf.shape}")
print(f"🏷️ Train labels' value counts: {y_train_rf.value_counts()}")

✂️ Train/test splitting for Random Forest dataset..
✅ Splitting is done successfully

Train size: (37251, 168)
Test size: (9313, 168)

📥 Saving test transaction IDs
✅Saved: [378233072  96311806 299045648]...

🛠️ Preparing Random Forest data...

📐 Train features' shape: (37251, 165)
🏷️ Train labels' value counts: is_illicit
0    33615
1     3636
Name: count, dtype: int64
